# Part 1: Import package

In [ ]:
pip install ag2

In [ ]:
pip install openai --upgrade pip

In [ ]:
pip install pyautogen

# Part 2: Agent configuation

In [ ]:
import autogen

In [ ]:
import os
K = os.environ["DEEPSEEK_API_KEY"]
llm_config = {
    "config_list": [
        # {
        #     "model": "GLM-4-Plus",
        #     "api_key": "",
        #     "base_url": "https://open.bigmodel.cn/api/paas/v4/",
        #     "price": [0, 0],
        #     "tags": ["glm"]
        # },        
        {
            "model": "deepseek-chat",
            "api_key": K,
            "base_url": "https://api.deepseek.com",
            "price": [0, 0],
            "tags": ["deepseek"]
        }
    ]
}

filtered_config = autogen.filter_config(
    config_list=llm_config["config_list"],
    filter_dict={"tags": ["deepseek"]}
)



# Part 3: Main part

In [ ]:
user= autogen.ConversableAgent(
    name= "user",
    llm_config= False,
    human_input_mode= "ALWAYS"
)

In [ ]:
def check_termination(message: dict)-> bool:
    """
    利用Agent判断对话是否结束
    """
    termination_agent= autogen.AssistantAgent(
        name= "众智",
        system_message= "You are Zhongzhi, an expert in emotion analysis responsible for deciding when a conversation should end. Reply with exactly 'END_CONVERSATION' if either condition below is met; otherwise, reply with exactly 'CONTINUE_CONVERSATION'. "
                        "Condition 1: The user no longer wants to continue the conversation. "
                        "Condition 2: The user asks for music recommendations. ",
        llm_config= {
            "cache_seed": None,
            **filtered_config[0]
        },
        human_input_mode= "NEVER"
    )
    user_input= message.get("content", "")
    termination_response= termination_agent.generate_reply(
        messages= [{"content": user_input, "role": "user"}]
    )
    return termination_response== "END_CONVERSATION"

In [ ]:



agent = autogen.ConversableAgent(
    name="千台",
    system_message="You are Qiantai, an expert conversational guide. Guide the user through a conversation, using Socratic questioning where appropriate, to learn about their mood, music preferences, and other relevant information. "
                     "When asked for a summary, summarize the conversation. ",
    human_input_mode="NEVER",
    is_termination_msg=check_termination ,
    llm_config={
        "cache_seed": None,
        **filtered_config[0]  # 关键：只传入一个模型配置！
    },
    
)

In [ ]:
def custom_summary(agent, recipient, summary_args):
    """
    自定义总结函数。
    """
    # 获取对话历史
    messages= agent.chat_messages[recipient]
    # 提取每条消息的内容
    contents = [msg["content"] for msg in messages]
    # 生成总结
    summary = "\n".join(contents)
    return f"Full conversation between the user and the conversational guide:\n{summary}"

In [ ]:
xinlifen= autogen.ConversableAgent(
    name= "新利芬",
    system_message= "You are Xinlifen, a psychologist responsible for developing a user profile based on the conversation between the user and the conversational guide. "
                    "A user profile is an abstract description of the user's characteristics, behavior, and preferences. "
                    "More specifically, try to include basic information (such as name, age, and gender), interests and preferences (such as music styles and artists), and psychological characteristics (such as lifestyle and values). "
                    "Do not recommend music to the user; your role is to analyze the user profile. ",
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    human_input_mode= "NEVER",
    description= "Develop a user profile" 
)

In [ ]:
xinlipei= autogen.ConversableAgent(
    name= "新利丕",
    system_message= "You are Xinlipi, a psychologist responsible for critically reviewing the user profile produced by Xinlifen. Check for missing information or omissions. "
                    "Do not recommend music to the user; your role is to analyze the user profile. "
                    "If the profile is rough because the user has not provided useful information, or if the profile needs no further revision, output 'PROFILE_COMPLETE'. "
                    "If you think the profile can be improved, output 'REFINE_PROFILE'. ",
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    human_input_mode= "NEVER",
    description= "Critically review the user profile"
)

In [ ]:
allowed_transitions= {
    xinlifen: [xinlipei],
    xinlipei: [xinlifen]
}

In [ ]:
xinli_group= autogen.GroupChat(
    agents= [xinlifen, xinlipei],
    speaker_transitions_type= "allowed",
    allowed_or_disallowed_speaker_transitions= allowed_transitions,
    messages= [],
    send_introductions= True
)

In [ ]:
xinli_group_manager= autogen.GroupChatManager(
    groupchat= xinli_group,
    system_message= "You coordinate the user-profiling group, organizing the two psychologists as they analyze the user profile. ",
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    is_termination_msg= lambda msg: "PROFILE_COMPLETE" in msg["content"]   
)

In [ ]:
tuijian= autogen.ConversableAgent(
    name= "推俭",
    system_message= "You are Tuijian, an expert music recommender. Recommend music based on the user profile, music preferences, and other relevant information. "
                    "Follow these instructions: 1. Consider both the user's music preferences and their current emotional state as reflected in the user profile. "
                    "2. Keep the recommendations focused instead of covering every music style. "
                    "3. Recommend two music styles, with three tracks for each style. "
                    "4. Include at least one and at most three links under each recommended track. Do not recommend a track if no link is available. "
                    "5. You may use the registered function to retrieve links for the recommended tracks. "
                    "6. When you finish recommending music, return 'RECOMMENDATIONS_COMPLETE'. ",
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    human_input_mode= "NEVER",
    description= "Expert music recommender"
)

# Part 4: LastFM music platform

In [ ]:
import requests
import json

In [ ]:
# 根据歌名搜索歌曲
def search_track(track_name, artist_name= None, limit= 5):
    url= "http://ws.audioscrobbler.com/2.0/"
    params= {
        "method": "track.search",
        "track": track_name,
        "api_key": os.environ["LASTFM_API_KEY"], # Last.fm API key
        "format": "json",
        "limit": limit
    }
    if artist_name:
        params["artist"]= artist_name
    response= requests.get(url, params= params)
    return response.json()

In [ ]:
def get_track_link(track_name: str, artist_name: str= None)-> json:
    music_search_agent= autogen.AssistantAgent(
        name= "专焕",
        system_message= "You are Zhuanhuan, a music expert with extensive knowledge of artists worldwide. "
                        "For the given artist and track, identify the artist's nationality. If the artist is Chinese, convert both the artist name and track title to Chinese. "
                        "If the artist is not Chinese, convert both the artist name and track title to English. Remember to convert the track title as well. "
                        "Output only a JSON object in the following format (example): {\"track_name\": \"漂洋过海来看你\", \"artist_name\": \"李宗盛\"}",
        llm_config= {
            "cache_seed": None,
            **filtered_config[0]
        },
        human_input_mode= "NEVER"
    )
    user_input= f"User-provided track title: {track_name}, Artist name: {artist_name}"
    search_result= music_search_agent.generate_reply(
        messages= [{"content": user_input, "role": "user"}]
    )
    # print(search_result)
    # Parse the search_result string into a dictionary
    try:
        search_result= json.loads(search_result)
    except json.JSONDecodeError:
        print("Failed to parse the search result as JSON.")
        return None
    # 
    music_name= search_result["track_name"]
    artist_name= search_result["artist_name"]
    search_result= search_track(music_name, artist_name)
    tracks= search_result.get("results", {}).get("trackmatches", {}).get("track", [])
    if not tracks:
        return
    # 获取前 3 个链接（如果不足 3 个，则全部输出）
    track_links= [track.get("url") for track in tracks[:3]]
    ret_json= {
        "music": music_name,
        "artist": artist_name,
        "links": track_links
    }
    return ret_json

In [ ]:
""" # 示例调用
track_name= "突然的自我"  # 歌名
artist_name= "伍佰"     # 艺术家名称（可选）
track_link= get_track_link(track_name, artist_name)
track_link """

# Part 5: Put together

In [ ]:
import os
from typing import Union
from typing import Literal

In [ ]:
user = autogen.UserProxyAgent(
    name="user",
    code_execution_config={
        "use_docker": False,  # 关键修改！禁用 Docker
        "work_dir": "coding"  # 代码执行目录
    }
)

In [ ]:
def check_termination(message: dict)-> bool:
    """
    利用Agent判断对话是否结束
    """
    termination_agent= autogen.AssistantAgent(
        name= "众智",
        system_message= "You are Zhongzhi, an expert in emotion analysis responsible for deciding when a conversation should end. Reply with exactly 'END_CONVERSATION' if either condition below is met; otherwise, reply with exactly 'CONTINUE_CONVERSATION'. "
                        "Condition 1: The user no longer wants to continue the conversation. "
                        "Condition 2: The user asks for music recommendations. ",
        llm_config= {
            "cache_seed": None,
            **filtered_config[0]
        },
        human_input_mode= "NEVER"
    )
    user_input= message.get("content", "")
    termination_response= termination_agent.generate_reply(
        messages= [{"content": user_input, "role": "user"}]
    )
    return termination_response== "END_CONVERSATION"

In [ ]:
qiantai= autogen.ConversableAgent(
    name= "千台",
    system_message= "You are Qiantai, an expert conversational guide. Guide the user through a conversation, using Socratic questioning where appropriate, to learn about their mood, music preferences, and other relevant information. "
                    "Do not recommend music. Your role is to guide the conversation and gather more information that will help with recommendations. ",
    is_termination_msg= check_termination,
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    human_input_mode= "NEVER" # Never ask for human input
)

In [ ]:
xinlipi= autogen.ConversableAgent(
    name= "新利丕",
    system_message= "You are Xinlipi, a psychologist responsible for critically reviewing the user profile produced by Xinlifen. Check for missing information or omissions. "
                    "Do not recommend music to the user; your role is to analyze the user profile. "
                    "Do not be overly critical; consider how much information the user has provided. "
                    "Note: If the profile is rough because the user has not provided useful information, or if the profile needs no further revision, output 'PROFILE_COMPLETE'. "    
                    "If you think the profile can be improved, identify areas for improvement based on the conversation and end with 'REFINE_PROFILE'. ",
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    human_input_mode= "NEVER",
    description= "Critically review the user profile"
)

In [ ]:
xinlifen= autogen.ConversableAgent(
    name= "新利芬",
    system_message= "You are Xinlifen, a psychologist responsible for developing a user profile based on the conversation between the user and the conversational guide. "
                    "A user profile is an abstract description of the user's characteristics, behavior, and preferences. "
                    "More specifically, try to include basic information (such as name, age, and gender), interests and preferences (such as music styles and artists), and psychological characteristics (such as lifestyle and values). "
                    "Try to include basic information (such as name, age, and gender), interests and preferences (such as music styles and artists), and psychological characteristics (such as lifestyle and values). "
                    "Even if the user asks you to recommend music, do not do so; your role is to analyze the user profile. Even if the user asks you to recommend music, do not do so; your role is to analyze the user profile. ",
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    human_input_mode= "NEVER",
    description= "Develop a user profile" 
)

In [ ]:
# The user proxy agent is used for interacting with the tuijian agent
# and executes tool calls.
tuoer= autogen.ConversableAgent(
    name= "tuoer",
    system_message= "You retrieve music links. You can execute the get_track_link function to obtain links for recommended tracks. Read the following instructions before calling it: ",
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    human_input_mode= "NEVER"
)

In [ ]:
tuijian= autogen.ConversableAgent(
    name= "推俭",
    system_message= "You are Tuijian, an expert music recommender. Recommend music based on the user profile, music preferences, and other relevant information. "
                    "Follow these instructions: 1. Consider both the user's music preferences and their current emotional state as reflected in the user profile. "
                    "2. Recommend two music styles, with three tracks for each style, unless the user specifies otherwise. "
                    "3. Try to avoid recommending tracks the user is already tired of hearing. "
                    "4. Use the registered function to retrieve links for the recommended tracks. ",
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    human_input_mode= "NEVER",
    description= "Expert music recommender"
)

In [ ]:
# Register the tool signature with the assistant agent.
tuijian.register_for_llm(name= "get_track_link", description="Retrieve links for a given track and artist.")(get_track_link)
# Register the tool function with the user proxy agent.
tuoer.register_for_execution(name= "get_track_link")(get_track_link)

In [ ]:
autogen.register_function(
    f= get_track_link,
    caller= tuijian,
    executor= tuoer,
    name= "get_track_link",
    description= "Retrieve music links. "
                 "Input: track title and artist name. "
                 "Output: track and artist information with up to three links."
)

In [ ]:

def custom_speaker_selection_func(
    last_speaker: autogen.Agent, 
    groupchat: autogen.GroupChat
) -> Union[autogen.Agent, Literal['auto', 'manual', 'random', 'round_robin'], None]:
    
    # 1. 确保 filtered_config 已定义（在函数外部定义）
    global filtered_config
    
    # 2. 提前定义决策agents（避免重复创建）
    if not hasattr(groupchat, '_decision_agents'):
        groupchat._decision_agents = [
            autogen.AssistantAgent(
                name=f"决策{i}",
                system_message="""You are an expert in language analysis responsible for routing the conversation:
                1. When the user requests music recommendations, reply with 'ROUTE_PROFILER'.
                2. When the user reports that a link is unavailable, reply with 'ROUTE_RECOMMENDER'.
                3. In other cases, reply with 'ROUTE_GUIDE'.
                4. When Profile Critic thinks the profile needs refinement, reply with 'ROUTE_PROFILER'.
                5. When Profile Critic considers the profile complete, reply with 'ROUTE_RECOMMENDER'.""",
                llm_config={
                    "cache_seed": None,
                    **filtered_config[0]  # 确保已正确定义
                },
                human_input_mode="NEVER"
            ) for i in range(1)
        ]

    # 3. 发言者判断逻辑
    agent_name = {
        user: "User",
        qiantai: "Conversational Guide",
        xinlifen: "Profile Builder", 
        xinlipi: "Profile Critic",
        tuijian: "Music Recommender"
    }.get(last_speaker, "Tool Executor")

    print(f'进入话语权判断函数！当前Speaker: {agent_name}')

    # 4. 决策流程
    if last_speaker in [user, xinlipi]:
        last_msg = groupchat.messages[-1]["content"]
        query = f"Speaker: {agent_name}, Content: {last_msg}"
        
        decisions = []
        for agent in groupchat._decision_agents:
            try:
                response = agent.generate_reply(
                    messages=[{"content": query, "role": "user"}]
                )
                decisions.append(response)
            except Exception as e:
                print(f"决策Agent出错: {str(e)}")
                decisions.append("ROUTE_GUIDE")  # 默认值

        # 投票统计
        from collections import Counter
        decision_counts = Counter(decisions)
        choice = decision_counts.most_common(1)[0][0]
        print(f'投票结果：{decision_counts}')

        return {
            "ROUTE_PROFILER": xinlifen,
            "ROUTE_RECOMMENDER": tuijian
        }.get(choice, qiantai)

    # 5. 其他转移逻辑
    transition_map = {
        qiantai: user,
        xinlifen: xinlipi,
        tuijian: tuoer
    }
    return transition_map.get(last_speaker, user)

In [ ]:
music_recommend_group= autogen.GroupChat(
    agents= [user, qiantai, xinlifen, xinlipi, tuijian, tuoer],
    speaker_selection_method= custom_speaker_selection_func,
    allow_repeat_speaker= False,
    messages=[],
    max_round=30
    
)

In [ ]:
music_recommend_group_manager= autogen.GroupChatManager(
    groupchat= music_recommend_group,
    system_message= "You manage the music recommendation group and are an expert in emotion analysis. Coordinate the conversational guide (who talks with the user), the profiling team (who analyze the user profile), the music recommender, and the link retrieval agent to provide music recommendations. ",
    llm_config= {
        "cache_seed": None,
        **filtered_config[0]
    },
    human_input_mode= "NEVER"
)

In [ ]:
sequential_chats= [
    {
        "recipient": qiantai,
        "summary_method": custom_summary,
    },
    {
        "recipient": music_recommend_group_manager,
        "message": "Please generate a profile for the user.",
        "summary_method": "reflection_with_llm"
    },
    {
        "recipient": tuijian,
        "message": "Please recommend music based on the user profile and music preferences.",
        "summary_method": "reflection_with_llm"
    }
]

In [ ]:
# chat_results= user.initiate_chats(sequential_chats)

In [ ]:

chat_results= user.initiate_chat(
    music_recommend_group_manager,
    summary_method= "last_msg",
    max_turns= 100
)
